### data collection and processing
To aquire the necessary song lyric data, [Genius' API](https://docs.genius.com) is used, because it could provide access to all lyrics from the initial selection. The process of obtaining lyrics is fairly easy, because there is only a client access token needed, which can be aquired with a free user account. The access token is locally stored in a [file](./key.json). To send queries to the API, the [lyricsgenius](https://pypi.org/project/lyricsgenius/) module is used, it returns the lyrics for a provided artist and songtitle as a string, which is then processed and stored together with other information for each song in a separate [file](./songlist_with_lyrics.json).

In [66]:
import re
import json
import string
import lyricsgenius
from retry import retry

In [67]:
# open the json-file containing all songs categorized by year and keyed with artist and title
with open('test_songs.json', 'r') as file:
    data = json.load(file)

In [68]:
# write an empty json file for all songs to be analized
with open('songlist_with_lyrics.json', 'w') as f:
    json.dump({'songs':[]}, f, indent=4)

In [69]:
with open('key.json', 'r') as file:
    genius_api = lyricsgenius.Genius(json.load(file)['apikey'])

In [70]:
# API call, queried by artist and title of each song
@retry(delay=5)
def fetch_from_api(artist, title):
    try:
        lyrics = genius_api.search_song(artist, title).lyrics
    except:
        print(f'ERROR: Timed out while fetching {title} by {artist} - trying it again...')
        raise
    return lyrics

In [71]:
# strip linebreaks '\n', punctuation and wrong words (like part names of songs) from song text provided by the API
def process_lyrics(lyrics):
    processed_lyrics = lyrics.lower()
    processed_lyrics = re.sub('\\[(.*?)\\]', '', processed_lyrics)

    translation_table_punct = str.maketrans('', '', string.punctuation)
    translation_table_linebreaks = str.maketrans('\n', ' ')
    processed_lyrics = processed_lyrics.translate(translation_table_punct).translate(translation_table_linebreaks)

    processed_lyrics = processed_lyrics.lstrip()
    processed_lyrics = processed_lyrics.replace('   ', ' ')
    processed_lyrics = processed_lyrics.replace('  ', ' ')
    return processed_lyrics

In [72]:
# function to append the json-string containing the lyrics to the already existing entries
def write_to_file(per_song_data):
    with open('songlist_with_lyrics.json', 'r') as f:
        file_data = json.load(f)
        file_data['songs'].append(per_song_data)

    with open('songlist_with_lyrics.json', 'w') as f:
        json.dump(file_data, f, indent=4)

In [73]:
# looping through the songs per year
for year in data:
    for song in range(len(data[year])):
        artist = data[year][song]['artist']
        title = data[year][song]['title']

        # append year to song's existing data
        year_dict = {'year': f"{year}"}
        data_per_song = data[year][song]
        data_per_song.update(year_dict)

        lyrics = fetch_from_api(artist, title)    
        
        # clean up the data from the API and update the content of the json-string
        lyrics = process_lyrics(lyrics)

        # lyrics = lyrics.encode('unicode_escape').decode('unicode_escape')

        # append the lyrics to the song's existing data
        lyrics_dict = {'lyrics': f"{lyrics}"}
        data_per_song.update(lyrics_dict)

        # write to 'songlist_with_lyrics.json'
        write_to_file(data_per_song)

ERROR: Timed out while fetching Billie Jean by Michael Jackson - trying it again...
ERROR: Timed out while fetching I Want to Know What Love Is by Foreigner - trying it again...
ERROR: Timed out while fetching Jump by Kris Kross - trying it again...


### lyrics analysis and categorization
During this process every song is analysed, word by word and compared via [longest common subsequence](https://en.wikipedia.org/wiki/Longest_common_subsequence) with a list of (root) words for each respective topic.

In [74]:
def lyrics_to_set(lyrics):
    words = lyrics.split()
    words = set(words)
    words = sorted(words)
    return words

In [75]:
def lyrics_to_list(lyrics):
    words = lyrics.split()
    return words

In [76]:
def longest_common_subsequence(word_from_lyric, word_from_topic, length_wfl, length_wft):
    if length_wfl == 0 or length_wft == 0:
        return 0

    if word_from_lyric[length_wfl - 1] == word_from_topic[length_wft - 1]:
        return 1 + longest_common_subsequence(word_from_lyric, word_from_topic, length_wfl - 1, length_wft - 1)

    else:
        return max(longest_common_subsequence(word_from_lyric, word_from_topic, length_wfl, length_wft - 1),
                   longest_common_subsequence(word_from_lyric, word_from_topic, length_wfl - 1, length_wft))

In [77]:
def lcs(word_from_lyric, word_from_topic):
    length_wfl = len(word_from_lyric)
    length_wft = len(word_from_topic)
    return longest_common_subsequence(word_from_lyric, word_from_topic, length_wfl, length_wft)

In [78]:
with open('songlist_with_lyrics.json', 'r') as file:
    data_songs = json.load(file)

In [83]:
with open('topics_choi_lee_hu_downie.json', 'r') as file:
    data_topics = json.load(file)

topic_dict = data_topics['topics'][0]
topics = topic_dict.keys()

In [80]:
# function to append the json-string containing the lyrics to the already existing entries
def update_file(per_song_data, index):
    with open('songlist_with_lyrics.json', 'r') as f:
        file_data = json.load(f)
        file_data['songs'][index].update(per_song_data)

    with open('songlist_with_lyrics.json', 'w') as f:
        json.dump(file_data, f, indent=4)

In [84]:
def compare_words_in_song(index):
    print(data_songs['songs'][index]['title'])
    current_song_lyrics = data_songs['songs'][index]['lyrics'].encode('unicode-escape').decode('unicode-escape')
    current_song_lyrics = lyrics_to_set(current_song_lyrics)

    lcs_counts_to_topics = {}

    for topic in topics:
        topic_root_words = topic_dict[topic]
        lcs_counts_to_topics[topic] = 0
        for root_word in topic_root_words:
            for word in current_song_lyrics:
                lcs_count = lcs(word, root_word)
                if lcs_count >= len(root_word):
                    current_count = lcs_counts_to_topics.get(topic)
                    lcs_counts_to_topics[topic] = current_count + 1

    print(lcs_counts_to_topics)
    sorted_dict = sorted(lcs_counts_to_topics, key=lcs_counts_to_topics.get, reverse=True)
    return sorted_dict[0]

In [85]:
for i in range(len(data_songs['songs'])):
    theme = compare_words_in_song(i)
    theme_dict = {'theme': f"{theme}"}
    data_per_song = data_songs['songs'][i]
    data_per_song.update(theme_dict)
    update_file(data_per_song, i)

Silly Love Songs
{'religion': 0, 'sex': 6, 'drugs': 0, 'parent': 3, 'war': 0, 'places': 1, 'ex-lover': 5, 'death': 0}
Don't Go Breaking My Heart
{'religion': 0, 'sex': 3, 'drugs': 0, 'parent': 3, 'war': 1, 'places': 3, 'ex-lover': 1, 'death': 3}
Disco Lady
{'religion': 4, 'sex': 12, 'drugs': 2, 'parent': 4, 'war': 1, 'places': 2, 'ex-lover': 3, 'death': 2}
December, 1963 (Oh, What a Night)
{'religion': 2, 'sex': 4, 'drugs': 1, 'parent': 2, 'war': 0, 'places': 1, 'ex-lover': 4, 'death': 2}
Play That Funky Music
{'religion': 3, 'sex': 18, 'drugs': 1, 'parent': 11, 'war': 1, 'places': 3, 'ex-lover': 2, 'death': 5}
Tonight's the Night (Gonna Be Alright)
{'religion': 1, 'sex': 7, 'drugs': 0, 'parent': 8, 'war': 2, 'places': 3, 'ex-lover': 5, 'death': 5}
I Just Want to Be Your Everything
{'religion': 4, 'sex': 9, 'drugs': 0, 'parent': 2, 'war': 1, 'places': 1, 'ex-lover': 6, 'death': 2}
Best of My Love
{'religion': 1, 'sex': 9, 'drugs': 1, 'parent': 6, 'war': 2, 'places': 2, 'ex-lover': 4, '